In [1]:
import numpy as np
import yfinance as yf
import pandas as pd

In [2]:
positions = {
    'KO': -125312,
    'PEP': 59350,
    'V': 30660,
    'MA': -18804,
    'XOM': -66386,
    'CVX': 54504,
    'JPM': 33003,
    'BAC': -189680
}

data = yf.download(list(positions.keys()), start='2024-02-12', end='2026-02-13', progress=False)['Close']
returns = data.pct_change().dropna()

C:\Users\samue\AppData\Local\Temp\ipykernel_18176\3793714959.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(list(positions.keys()), start='2024-02-12', end='2026-02-13', progress=False)['Close']


In [3]:
latest_prices = data.iloc[-1]

# Valor de cada posición
values = pd.Series(positions) * latest_prices

# Valor total del portafolio
portfolio_value = sum([abs(x) for x in values.tolist()])

# Pesos de cada activo
weights = values / portfolio_value
weights = weights.to_frame(name='Peso (%)')
weights * 100

,Peso (%)
BAC,-12.526793
CVX,12.501060
JPM,12.559531
KO,-12.448396
MA,-12.471924
PEP,12.478163
V,12.498331
XOM,-12.515802


In [4]:
# Retornos del portafolio
portfolio_returns = returns.dot(weights)

# P&L
portfolio_pnl = portfolio_returns * portfolio_value

VaR_95 = np.percentile(portfolio_pnl, 5)
print("VaR histórico 1 día 95%: ", VaR_95)
print("VaR histórico 1 día 95% en $: ", VaR_95 * portfolio_value)

VaR histórico 1 día 95%:  -315351.3491271099
VaR histórico 1 día 95% en $:  -25078470088111.387


In [5]:
volume = yf.download(list(positions.keys()), start='2025-11-12', end='2026-02-13', progress=False)['Volume']
avg_volume = volume.mean()

C:\Users\samue\AppData\Local\Temp\ipykernel_18176\329622128.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  volume = yf.download(list(positions.keys()), start='2025-11-12', end='2026-02-13', progress=False)['Volume']


In [7]:
liquidity_check = {}

for ticker in positions.keys():
    shares = abs(positions[ticker])
    max_daily_trade = 0.10 * avg_volume[ticker]
    days_needed = shares / max_daily_trade
    liquidity_check[ticker] = days_needed

liquidity = pd.DataFrame.from_dict(liquidity_check, orient='index', columns=['Days to Liquidate'])
liquidity['Cumple ?'] = liquidity['Days to Liquidate'] < 2

liquidity

,Days to Liquidate,Cumple ?
KO,0.069788,True
PEP,0.070315,True
V,0.041600,True
MA,0.055088,True
XOM,0.036447,True
CVX,0.051773,True
JPM,0.031439,True
BAC,0.049431,True
